In [38]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 00_rq1_qual_analysis.py
# Purpose of Script: Identify all synthetic media in Qualitative SOR data.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Install Packages
#~~~~~~~~~~~~~~~~~~~~~~~~~~
!pip install langdetect
!pip install deep_translator

In [39]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import gc
import duckdb
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
from deep_translator import GoogleTranslator

In [40]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [41]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect()

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/00_backups/01_sample_backups/00_20260629/"
path_data = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/00_backups/00_data_backups/02_data_backup_20260630/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [42]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Functions ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Language Detection
def detect_language(text):
  if not isinstance(text, str) or text.strip() == "":
    return "No Language"
  try:
    return detect(text)
  except LangDetectException:
    return "No Language"


In [51]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Qualitative Analysis ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Motivation: Search Qualitative Analysis for Synthethic Media
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_qual = con.execute(f""" select * from '{path_samp}qual.parquet'""").df()

In [52]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Find Unique Q/Statement Combinations ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_qual_cut = df_qual[["platform","q","statement"]].drop_duplicates()

# Frequency Table
### Number of Unique Statements per Qualitative Statement per Platform
out_qual_freq = (df_qual_cut.groupby(['platform','q'])['statement']
                 .nunique()
                 .reset_index(name = 'n_unique_statements'))

### Frequencies of Statements
out_qual_freq_state = (df_qual_cut.groupby(['platform','q','statement'])
                 .size()
                 .reset_index(name = 'count'))

In [59]:
df_qual_cut

,platform,q,statement
0,facebook,incompatible_content_ground,"this was a violation of """"sections 3.2"""" of ou..."
1,facebook,incompatible_content_ground,
2,facebook,incompatible_content_explanation,this was decision_visibility_content_removed b...
3,facebook,incompatible_content_explanation,this was decision_account_suspended because it...
4,facebook,incompatible_content_explanation,this was decision_visibility_content_labelled ...
...,...,...,...
384683,youtube,illegal_content_explanation,"hi [redacted], we have received a trademark c..."
385262,youtube,decision_facts,<p>content that shows certain illegal or regul...
385810,youtube,decision_facts,"hi [redacted], channel: [redacted] <https://w..."
385885,youtube,illegal_content_explanation,"hi [redacted], channel: [redacted] <https://w..."


In [53]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Output Unique Statements
#~~~~~~~~~~~~~~~~~~~~~~~~~~
out_qual_freq.to_csv(f"""{path_out}00_qualitative_statement_frequency.csv""")
out_qual_freq_state.to_csv(f"""{path_out}00_qualtative_statement_language_frequency.csv""")

In [55]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Detect Languages ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Obtain all statements in English
### Define Statement Language
out_qual_freq_state["statement_lang"] = out_qual_freq_state["statement"].apply(detect_language)

In [56]:
# Assess distribution of languages - ALL
lang_freq = pd.DataFrame(out_qual_freq_state["statement_lang"].value_counts())

# Assess distribution of languages - Platform Level
lang_freq_plat = (out_qual_freq_state.groupby(['platform','statement_lang']).size().reset_index(name='count'))
lang_freq_plat_cross = pd.crosstab(lang_freq_plat["platform"], lang_freq_plat["statement_lang"])

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Translate Statements ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Isolate Subset to Translate
df_to_translate = out_qual_freq_state[(out_qual_freq_state['statement_lang'] != "No Language") & (out_qual_freq_state['statement_lang'] != 'en')]
df_to_translate = df_to_translate.sort_values(['platform','statement_lang'], ascending=[True, False]).reset_index(drop=True)

In [ ]:
# Manually Remove Detection False Positives
### Remove all X entries (strings are concatenated english)
df_to_translate = df_to_translate[df_to_translate['platform'] != 'x']

### Remove all Snapchat entries (small strings are all english but not correctly identified as such)
df_to_translate = df_to_translate[df_to_translate['platform'] != 'snapchat']

In [ ]:
# Translate
translator = GoogleTranslator(source = "auto", target = "en")
df_to_translate['statement_en'] = df_to_translate['statement'].apply(lambda x: translator.translate(x))

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Translated Data
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_to_translate.to_csv(f"""{path_out}00_qualitative_analysis_translate.csv""")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Synthetic Media Search ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# need to search for content that has the following in the string
# 'synthetic' - x and tiktok
# 'syn' - x and tiktok
# 'manipulated' - youtube (2 entries)/ 'technically manipulated' - 'manipulate - 1 more entry'
# 'generative' - snapchat - 2 entries
# 'artificial' - tiktok - 11 entries
# 'deepfake' - 0 entries (deep = 13 from tiktok but not relevant)
# 'digitally created' - tiktok

In [46]:
# Import Translated Unique Data
df_to_translate = pd.read_csv(f"""{path_out}00_qualitative_analysis_translate.csv""")

In [ ]:
# Join Translations to Unique Statments
df = out_qual_freq_state
df = df.drop(columns = ['count'])
df = df.merge(df_to_translate.drop(columns = ['count']), on=['platform','q','statement','statement_lang'], how = 'left')
df['statement_final'] = df['statement_en'].fillna(df['statement'])
df = df[["platform","q","statement","statement_final"]]

In [ ]:
df.head(100)

,platform,q,statement,statement_final
0,facebook,category_specification_other,0,0
1,facebook,decision_facts,this account was determined to violate terms o...,this account was determined to violate terms o...
2,facebook,decision_facts,this content was determined to violate terms o...,this content was determined to violate terms o...
3,facebook,decision_facts,this service was determined to violate terms o...,this service was determined to violate terms o...
4,facebook,decision_visibility_other,0,0
...,...,...,...,...
95,snapchat,incompatible_content_explanation,see explanation here: https://snap.com/content...,see explanation here: https://snap.com/content...
96,snapchat,incompatible_content_explanation,see explanation here: https://snap.com/content...,see explanation here: https://snap.com/content...
97,snapchat,incompatible_content_explanation,see explanation here: https://snap.com/content...,see explanation here: https://snap.com/content...
98,snapchat,incompatible_content_explanation,see explanation here: https://snap.com/content...,see explanation here: https://snap.com/content...
